<a href="https://colab.research.google.com/github/falyseck/text_classification/blob/main/text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SDG 3 Indicator Multi-Label Text Classification
**Group Assignment 2** — Complete pipeline: EDA → Preprocessing → Feature Engineering → Experiments → Evaluation → Inference


### Sections
1. Setup & Installs
2. Load Data
3. Exploratory Data Analysis (EDA)
4. Preprocessing Pipeline
5. Feature Engineering
6. Experiments (8 total)
7. Results Summary & Model Comparison
8. Inference on Test Set


---
## Section 1 — Setup & Installs


In [ ]:
# Install required libraries (only needed once per Colab session)
!pip install -q scikit-multilearn imbalanced-learn sentence-transformers seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 2.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# NLP
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import hamming_loss, classification_report, f1_score
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

# scikit-multilearn
from skmultilearn.model_selection import iterative_train_test_split

# Sentence Transformers
from sentence_transformers import SentenceTransformer

# Scipy
from scipy.sparse import hstack, issparse

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


print('All libraries loaded successfully.')

All libraries loaded successfully.


---
## Section 2 — Load Data


In [ ]:

# ── Load datasets


TRAIN_PATH = 'Devex_train.csv'
TEST_PATH  = 'Devex_test_questions.csv'
def read_csv_safe(path):
    # Define a comprehensive list of values to be interpreted as NaN
    na_vals = ['', '#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan', \
               '1.#IND', '1.#QNAN', '<NA>', 'N/A', 'NA', 'NULL', 'NaN', 'n/a', 'nan', 'null']
    for enc in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
        try:
            return pd.read_csv(path, encoding=enc, na_values=na_vals, keep_default_na=True)
        except UnicodeDecodeError:
            continue
    raise ValueError(f'Could not read {path} with any common encoding')

train_df = read_csv_safe(TRAIN_PATH)
test_df  = read_csv_safe(TEST_PATH)

print('Train shape:', train_df.shape)
print('Test  shape:', test_df.shape)
print()
print('Train columns:', train_df.columns.tolist())
print('Test  columns:', test_df.columns.tolist())

Train shape: (2995, 15)
Test  shape: (998, 3)

Train columns: ['Unique ID', 'Type', 'Text', 'Label 1', 'Label 2', 'Label 3', 'Label 4', 'Label 5', 'Label 6', 'Label 7', 'Label 8', 'Label 9', 'Label 10', 'Label 11', 'Label 12']
Test  columns: ['Unique ID', 'Type', 'Text']


In [ ]:
# Preview the training data
train_df.head(3)

,Unique ID,Type,Text,Label 1,Label 2,Label 3,Label 4,Label 5,Label 6,Label 7,Label 8,Label 9,Label 10,Label 11,Label 12
0,12555,Grant,Centers of Biomedical Research Excellence (COB...,3.b.2 - Total net official development assista...,3.c.1 - Health worker density and distribution,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,14108,Grant,Research on Regenerative Medicine <h2><strong>...,3.b.2 - Total net official development assista...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,23168,Organization,Catholic Health Association of India (CHAI): <...,3.d.1 - International Health Regulations (IHR)...,3.8.1 - Coverage of essential health services ...,3.8.2 - Proportion of population with large ho...,3.b.3 - Proportion of health facilities that h...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ── Identify the text column and label columns

TEXT_COL = 'Text'
LABEL_COLS = [f'Label {i}' for i in range(1, 11)]

# Convert identified label columns to binary (1 if present, 0 if missing/empty)
for col in LABEL_COLS:
    # This function will handle explicit NaN, None, empty strings, and common string representations of missing data
    def is_label_present(val):
        # Explicitly check for actual NaN or None
        if pd.isna(val) or val is None:
            return 0
        # If it's a string, check for empty string after strip, or common "missing" strings
        if isinstance(val, str):
            val_lower = val.strip().lower()
            if val_lower == '' or val_lower in ['nan', 'none', 'null', 'n/a', '-']:
                return 0
        # Otherwise, consider it a present label
        return 1
    train_df[col] = train_df[col].apply(is_label_present)

print(f'Text column  : {TEXT_COL}')
print(f'Label columns ({len(LABEL_COLS)}): {LABEL_COLS[:5]} ...')
print()
print('Sample text:')
print(train_df[TEXT_COL].iloc[0][:300])

Text column  : Text
Label columns (10): ['Label 1', 'Label 2', 'Label 3', 'Label 4', 'Label 5'] ...

Sample text:
Centers of Biomedical Research Excellence (COBRE) Phase III - Transitional Centers     <p><strong>Funding Opportunity Description</strong></p>    <p><a name="_Toc258873267"></a>The Institutional Development Award (IDeA) Program endeavors to stimulate research at institutions in states that have not 


In [ ]:
# Basic data quality checks
print('Missing values in train:')
print(train_df.isnull().sum())
print()
print('Missing values in test:')
print(test_df.isnull().sum())

# Fill any missing text with empty string
train_df[TEXT_COL] = train_df[TEXT_COL].fillna('')
test_df[TEXT_COL]  = test_df[TEXT_COL].fillna('')

print('\nMissing values handled.')

Missing values in train:
Unique ID       0
Type            0
Text            0
Label 1         0
Label 2         0
Label 3         0
Label 4         0
Label 5         0
Label 6         0
Label 7         0
Label 8         0
Label 9         0
Label 10        0
Label 11     2995
Label 12     2995
dtype: int64

Missing values in test:
Unique ID    0
Type         0
Text         0
dtype: int64

Missing values handled.


## SECTION 3 : Label frequency distribution


In [ ]:
# ── 3.1 Label frequency distribution ──────────────────────────────────────────
label_counts = train_df[LABEL_COLS].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
label_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Label frequency — SDG 3 indicators', fontsize=14)
ax.set_xlabel('Indicator')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('eda_label_frequency.png', dpi=150)
plt.show()

print('Most common labels:')
print(label_counts.head(5))
print('\nLeast common labels:')
print(label_counts.tail(5))

In [ ]:
# ── 3.2 Labels per sample distribution ─────────────────────────────────────────
labels_per_sample = train_df[LABEL_COLS].sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cast the max value to int as range() expects integers
labels_per_sample.hist(bins=range(0, int(labels_per_sample.max())+2), ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Labels per sample')
axes[0].set_xlabel('Number of labels')
axes[0].set_ylabel('Samples')

# Text length distribution
text_lengths = train_df[TEXT_COL].str.split().str.len()
text_lengths.hist(bins=50, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Text length (words)')
axes[1].set_xlabel('Word count')
axes[1].set_ylabel('Samples')

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150)
plt.show()

print(f'Avg labels per sample : {labels_per_sample.mean():.2f}')
print(f'Max labels per sample : {labels_per_sample.max()}')
print(f'Samples with 0 labels : {(labels_per_sample == 0).sum()}')
print(f'Avg text length (words): {text_lengths.mean():.0f}')

In [ ]:
# ── 3.3 Label co-occurrence heatmap ───────────────────────────────────────────
label_matrix = train_df[LABEL_COLS].values
co_occur = label_matrix.T @ label_matrix  # shape: (n_labels, n_labels)
co_occur_df = pd.DataFrame(co_occur, index=LABEL_COLS, columns=LABEL_COLS)

# Normalise by diagonal (self-count) to get conditional probability
diag = np.diag(co_occur).astype(float)
diag[diag == 0] = 1
co_occur_norm = co_occur / diag[:, None]
co_occur_norm_df = pd.DataFrame(co_occur_norm, index=LABEL_COLS, columns=LABEL_COLS)
np.fill_diagonal(co_occur_norm_df.values, 0)  # hide self-co-occurrence

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(co_occur_norm_df, cmap='Blues', ax=ax, square=True,
            linewidths=0.3, cbar_kws={'label': 'Conditional co-occurrence'})
ax.set_title('Label co-occurrence heatmap (normalised)', fontsize=13)
plt.tight_layout()
plt.savefig('eda_cooccurrence.png', dpi=150)
plt.show()

# Baseline Hamming Loss (always predict 0)
y_all = train_df[LABEL_COLS].values
zero_pred = np.zeros_like(y_all)
baseline_hl = hamming_loss(y_all, zero_pred)
print(f'Baseline Hamming Loss (all zeros): {baseline_hl:.4f}')
print('This is the floor — any model must beat this.')

In [ ]:
## SECTION 4 : 'Model Building and Evaluation'

In [ ]:
# ── Preprocessing function ─────────────────────────────────────────────────────
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# SDG-domain stopwords to keep (they carry meaning in this domain)
KEEP_WORDS = {'health', 'disease', 'mortality', 'maternal', 'child', 'mental',
              'nutrition', 'water', 'sanitation', 'tobacco', 'vaccine', 'hiv',
              'aids', 'malaria', 'tuberculosis', 'drug', 'sexual', 'reproductive'}
stop_words -= KEEP_WORDS

def preprocess_text(text, lemmatize=True, remove_stopwords=True):
    """Clean and normalise a single text string."""
    # Lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenise
    tokens = word_tokenize(text)
    # Remove stopwords
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # Lemmatise
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Apply to train and test
print('Preprocessing train texts...')
train_df['text_clean'] = train_df[TEXT_COL].apply(preprocess_text)
print('Preprocessing test texts...')
test_df['text_clean']  = test_df[TEXT_COL].apply(preprocess_text)

print('Done.')
print('\nSample original:')
print(train_df[TEXT_COL].iloc[0][:200])
print('\nSample cleaned:')
print(train_df['text_clean'].iloc[0][:200])

In [ ]:
# ── Train / validation split (stratified for multi-label) ─────────────────────
from scipy.sparse import csr_matrix

X_text = train_df['text_clean'].values
Y      = train_df[LABEL_COLS].values.astype(int)

# iterative_train_test_split requires dense arrays as index arrays
indices = np.arange(len(X_text)).reshape(-1, 1)
idx_train, _, idx_val, _ = iterative_train_test_split(
    indices, Y, test_size=0.2
)
idx_train = idx_train.ravel()
idx_val   = idx_val.ravel()

X_train_text = X_text[idx_train]
X_val_text   = X_text[idx_val]
Y_train      = Y[idx_train]
Y_val        = Y[idx_val]
X_test_text  = test_df['text_clean'].values

print(f'Train size : {len(X_train_text)}')
print(f'Val   size : {len(X_val_text)}')
print(f'Test  size : {len(X_test_text)}')

# Helper to evaluate any prediction
def evaluate(y_true, y_pred, name='Model'):
    hl  = hamming_loss(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average='macro',  zero_division=0)
    f1w = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    print(f'{name:40s}  HL={hl:.4f}  F1-macro={f1m:.4f}  F1-weighted={f1w:.4f}')
    return {'name': name, 'hamming_loss': hl, 'f1_macro': f1m, 'f1_weighted': f1w}

results = []  # collect all experiment results


## Section 5 — Feature Engineering


In [ ]:
# ── 5.1 TF-IDF features ────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=30000,
    sublinear_tf=True,
    min_df=2
)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf   = tfidf.transform(X_val_text)
X_test_tfidf  = tfidf.transform(X_test_text)

print('TF-IDF feature matrix shape:', X_train_tfidf.shape)

In [ ]:
# ── 5.2 Sentence-BERT embeddings (used in Experiments 4, 7, 8)


print('Loading sentence-transformers model...')
sbert = SentenceTransformer('all-MiniLM-L6-v2')

print('Encoding train texts...')
X_train_sbert = sbert.encode(X_train_text.tolist(), batch_size=64,
                              show_progress_bar=True, convert_to_numpy=True)
print('Encoding val texts...')
X_val_sbert   = sbert.encode(X_val_text.tolist(), batch_size=64,
                              show_progress_bar=True, convert_to_numpy=True)
print('Encoding test texts...')
X_test_sbert  = sbert.encode(X_test_text.tolist(), batch_size=64,
                              show_progress_bar=True, convert_to_numpy=True)

print('\nSBERT embedding shape:', X_train_sbert.shape)

---
## Section 6 — Experiments
Each cell is one experiment.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 1 — TF-IDF + Logistic Regression (BASELINE)
# Why: Establish a simple sparse-feature baseline.
# ══════════════════════════════════════════════════════════════════════════════
clf_exp1 = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE),
    n_jobs=-1
)
clf_exp1.fit(X_train_tfidf, Y_train)
pred_exp1 = clf_exp1.predict(X_val_tfidf)

res = evaluate(Y_val, pred_exp1, 'Exp 1: TF-IDF + Logistic Regression')
results.append(res)

Exp 1: TF-IDF + Logistic Regression       HL=0.0760  F1-macro=0.1697  F1-weighted=0.6864


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 2 — TF-IDF + Random Forest
# Why: Test a tree-based ensemble on the same sparse features as Exp 1.
#      Does non-linearity help over logistic regression?
# ══════════════════════════════════════════════════════════════════════════════
clf_exp2 = OneVsRestClassifier(
    RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
)
clf_exp2.fit(X_train_tfidf, Y_train)
pred_exp2 = clf_exp2.predict(X_val_tfidf)

res = evaluate(Y_val, pred_exp2, 'Exp 2: TF-IDF + Random Forest')
results.append(res)

Exp 2: TF-IDF + Random Forest             HL=0.0681  F1-macro=0.2489  F1-weighted=0.7723


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 3 — TF-IDF + LinearSVC
# Why: LinearSVC is often strong on high-dimensional sparse text. Compare to LR.
# ══════════════════════════════════════════════════════════════════════════════
clf_exp3 = OneVsRestClassifier(
    LinearSVC(max_iter=2000, C=0.5, random_state=RANDOM_STATE),
    n_jobs=-1
)
clf_exp3.fit(X_train_tfidf, Y_train)
pred_exp3 = clf_exp3.predict(X_val_tfidf)

res = evaluate(Y_val, pred_exp3, 'Exp 3: TF-IDF + LinearSVC')
results.append(res)

Exp 3: TF-IDF + LinearSVC                 HL=0.0805  F1-macro=0.1624  F1-weighted=0.6547
